In [ ]:
"""
3-Stage Chest X-ray Pipeline - GUI (Gradio)
Stage 1 SqueezeNet  : Normal vs Abnormal
Stage 2 ResNet-50   : COVID-19 vs Non-COVID
Stage 3 EfficientNet: Pneumonia Bacteria / Pneumonia Virus / Tuberculosis

Needs ONLY the 3 trained .pt files - no dataset, no training, no other notebook cells.

Usage (Colab):   !pip install -q gradio ; %run cxr_gui_app.py
Usage (own PC):  pip install torch torchvision gradio pandas matplotlib pillow ; python cxr_gui_app.py
"""
import os, sys, json, warnings
import numpy as np
import pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import models
import matplotlib
try:
    from IPython import get_ipython
    _in_nb = get_ipython() is not None
except Exception:
    _in_nb = False
if not _in_nb:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import gradio as gr

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =====================================================================================
# CONFIG  (edit only this block)
# =====================================================================================
# folders searched (in order) for the .pt weight files
_APP_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
WEIGHT_DIRS = ["results", "weights", ".", "/content/results",
               "/content/drive/MyDrive/CXR_pipeline/results", "/content/drive/MyDrive/CXR_pipeline",
               os.path.join(_APP_DIR, "results"), os.path.join(_APP_DIR, "weights"), _APP_DIR]

# "v2" = optimised Stage 3 (320 px, file stage3v2_efficientnet_b0_best.pt, test acc 81.71 %)
# "v1" = first Stage 3     (224 px, file stage3_EfficientNet-b0_best.pt,   test acc 83.88 %)
STAGE3_VERSION = "v2"

# ---- ALL METRICS (%), paper vs ours.  Saved to metrics.json next to the weights on first run;
#      edit that json any time to type the paper's Table-4 values (None / null = shown as "-").
METRIC_KEYS = ["Accuracy", "Precision", "Recall", "F1-score", "Specificity", "AUC"]
DEFAULT_METRICS = {
    "paper": {   # Soft Computing 2026. Accuracy = abstract. Recall = 1 - FNR (FNR quoted in paper text, Sec. 7).
                 # Precision / F1 / Specificity / AUC are in the paper's Table 4 -> type them into metrics.json.
        "1": {"Accuracy": 99.0, "Precision": None, "Recall": 97.9, "F1-score": None, "Specificity": None, "AUC": None},
        "2": {"Accuracy": 98.0, "Precision": None, "Recall": 97.8, "F1-score": None, "Specificity": None, "AUC": None},
        "3": {"Accuracy": 97.0, "Precision": None, "Recall": 97.26, "F1-score": None, "Specificity": None, "AUC": None},
    },
    "ours": {    # from your notebook outputs (test set; Stage 3 = macro average)
        "1":   {"Accuracy": 92.32, "Precision": 97.66, "Recall": 90.96, "F1-score": 94.19, "Specificity": 95.27, "AUC": 97.77},
        "2":   {"Accuracy": 99.02, "Precision": 99.66, "Recall": 98.38, "F1-score": 99.02, "Specificity": 99.67, "AUC": 99.98},
        "3v1": {"Accuracy": 83.88, "Precision": 86.00, "Recall": 87.21, "F1-score": 86.45, "Specificity": None, "AUC": 93.43},
        "3v2": {"Accuracy": 81.71, "Precision": 84.36, "Recall": 82.84, "F1-score": 83.52, "Specificity": None, "AUC": 91.92},
    },
    "ours_stage3_per_class": {   # test set, precision / recall / f1 (%)
        "3v1": {"Pneumonia Bacteria": [88.86, 82.39, 85.51], "Pneumonia Virus": [69.14, 79.25, 73.85], "Tuberculosis": [100.0, 100.0, 100.0]},
        "3v2": {"Pneumonia Bacteria": [82.41, 86.85, 84.57], "Pneumonia Virus": [70.68, 63.68, 67.00], "Tuberculosis": [100.0, 98.00, 98.99]},
    },
    "train_val_test_acc": {
        "1": [94.06, 93.30, 92.32], "2": [99.83, 99.05, 99.02],
        "3v1": [83.29, 79.92, 83.88], "3v2": [90.49, 82.13, 81.71],
    },
}

STAGES = {
    1: dict(name="SqueezeNet",      arch="squeezenet", size=224, classes=["Abnormal", "Normal"],
            files=["stage1_SqueezeNet_best.pt"]),
    2: dict(name="ResNet-50",       arch="resnet50",   size=224, classes=["COVID-19", "Non-COVID"],
            files=["stage2_ResNet-50_best.pt"]),
    3: dict(name="EfficientNet-b0", arch="effnet_b0",  size=320 if STAGE3_VERSION == "v2" else 224,
            classes=["Pneumonia Bacteria", "Pneumonia Virus", "Tuberculosis"],
            files=(["stage3v2_efficientnet_b0_best.pt"] if STAGE3_VERSION == "v2"
                   else ["stage3_EfficientNet-b0_best.pt"])),
}

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


# =====================================================================================
# MODELS  (same heads as training, so the saved state_dicts load exactly)
# =====================================================================================
def build_model(arch, n_classes):
    if arch == "squeezenet":
        m = models.squeezenet1_1(weights=None)
        m.classifier = nn.Sequential(nn.Dropout(0.5), nn.Conv2d(512, n_classes, 1),
                                     nn.AdaptiveAvgPool2d(1), nn.Flatten())
    elif arch == "resnet50":
        m = models.resnet50(weights=None)
        m.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(m.fc.in_features, n_classes))
    elif arch == "effnet_b0":
        m = models.efficientnet_b0(weights=None)
        m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.classifier[1].in_features, n_classes))
    else:
        raise ValueError(arch)
    return m


def _mount_drive_if_possible():
    try:
        from google.colab import drive
        if not os.path.isdir("/content/drive/MyDrive"):
            drive.mount("/content/drive")
    except Exception:
        pass


def _find(fname):
    for d in WEIGHT_DIRS:
        p = os.path.join(d, fname)
        if os.path.isfile(p):
            return p
    return None


def load_pipeline():
    if any(_find(f) is None for s in STAGES.values() for f in s["files"]):
        _mount_drive_if_possible()
    pipe, missing = {}, []
    for k, s in STAGES.items():
        path = _find(s["files"][0])
        if path is None:
            missing.append(s["files"][0]); continue
        m = build_model(s["arch"], len(s["classes"]))
        m.load_state_dict(torch.load(path, map_location="cpu"))
        pipe[k] = m.to(device).eval()
        print(f"Stage {k}: {s['name']:<16} loaded from {path}")
    if missing:
        raise FileNotFoundError(
            "Weight file(s) not found: " + ", ".join(missing) +
            "\nSearched in: " + ", ".join(WEIGHT_DIRS) +
            "\n-> copy the .pt files into a 'results' folder (or Google Drive: MyDrive/CXR_pipeline/results).")
    return pipe


PIPE = load_pipeline()


def load_metrics():
    """metrics.json (if present) overrides the defaults key-by-key; if absent it is created next to the weights."""
    import copy
    m = copy.deepcopy(DEFAULT_METRICS)
    wdir = os.path.dirname(_find(STAGES[1]["files"][0]))
    path = os.path.join(wdir, "metrics.json")
    if os.path.isfile(path):
        try:
            user = json.load(open(path))
            for sec in ("paper", "ours"):
                for k, d in user.get(sec, {}).items():
                    m[sec].setdefault(k, {}).update(d)
            for sec in ("ours_stage3_per_class", "train_val_test_acc"):
                m[sec].update(user.get(sec, {}))
            print("metrics loaded from", path)
        except Exception as e:
            print("could not read", path, "->", e, "(using defaults)")
    else:
        try:
            json.dump(DEFAULT_METRICS, open(path, "w"), indent=2)
            print("metrics.json created at", path, "(edit it to add paper Table-4 values)")
        except Exception:
            pass
    return m


METRICS = load_metrics()
S3 = "3" + STAGE3_VERSION
def _m(who, stage, key):
    return METRICS[who].get(S3 if stage == 3 and who == "ours" else str(stage), {}).get(key)
def _fmt(v):
    return "-" if v is None else f"{v:.2f}%"
CAM_LAYERS = {1: PIPE[1].features[-1], 2: PIPE[2].layer4[-1], 3: PIPE[3].features[-1]}


# =====================================================================================
# PREPROCESSING + INFERENCE + GRAD-CAM
# =====================================================================================
def preprocess(pil_img, size):
    """Same as training: grayscale -> resize -> 3 channels -> ImageNet normalisation."""
    im = pil_img.convert("L").resize((size, size), Image.BILINEAR)
    x = torch.from_numpy(np.asarray(im, dtype=np.uint8)).unsqueeze(0).float().div(255).expand(3, -1, -1)
    return ((x - MEAN) / STD).unsqueeze(0).to(device), im


def grad_cam(model, layer, x, class_idx):
    store = {}

    def fwd_hook(module, inp, out):
        store["a"] = out.detach()
        out.register_hook(lambda g: store.__setitem__("g", g.detach()))

    h = layer.register_forward_hook(fwd_hook)
    try:
        model.eval(); model.zero_grad(set_to_none=True)
        with torch.enable_grad():
            logits = model(x.clone().float().requires_grad_(True))
            logits[0, class_idx].backward()
    finally:
        h.remove()
    a, g = store["a"][0], store["g"][0]
    cam = F.relu((g.mean(dim=(1, 2), keepdim=True) * a).sum(0))
    cam = cam / (cam.max() + 1e-8)
    cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    return cam.cpu().numpy()


def _fig_to_array(fig):
    fig.canvas.draw()
    arr = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
    plt.close(fig)
    return arr


def _cam_figure(inputs, cams, title, color):
    n = 1 + len(cams)
    fig, ax = plt.subplots(1, n, figsize=(4.6 * n, 5))
    ax = np.atleast_1d(ax)
    ax[0].imshow(inputs, cmap="gray"); ax[0].set_title("Input (as seen by the model)", fontsize=10); ax[0].axis("off")
    for a_, (t, im, cam) in zip(ax[1:], cams):
        a_.imshow(im, cmap="gray"); a_.imshow(cam, cmap="jet", alpha=0.45, vmin=0, vmax=1)
        a_.contour(cam, levels=[0.6], colors="white", linewidths=1.3)
        a_.set_title(t, fontsize=10); a_.axis("off")
    fig.suptitle(title, fontsize=13, color=color)
    fig.tight_layout()
    return _fig_to_array(fig)


def predict(pil_img):
    if pil_img is None:
        return "### Upload a chest X-ray first.", None, None, None
    pil_img = pil_img.convert("RGB")
    rows, cams = [], []

    def run(k, class_idx=None):
        x, im = preprocess(pil_img, STAGES[k]["size"])
        with torch.no_grad():
            p = F.softmax(PIPE[k](x).float(), 1)[0].cpu().numpy()
        return x, im, p

    def row(k, cls, conf):
        return [f"Stage {k}", STAGES[k]["name"], cls, f"{conf*100:.1f}%",
                _fmt(_m("paper", k, "Accuracy")), _fmt(_m("ours", k, "Accuracy")),
                _fmt(_m("paper", k, "F1-score")), _fmt(_m("ours", k, "F1-score"))]

    x1, im1, p1 = run(1); i1 = int(p1.argmax()); c1 = STAGES[1]["classes"][i1]
    rows.append(row(1, c1, p1[i1]))
    last_probs, last_classes = p1, STAGES[1]["classes"]
    if c1 == "Normal":
        final, conf = "Normal - no abnormality detected", p1[i1]
    else:
        cams.append(("Stage 1: where the model sees abnormality", im1, grad_cam(PIPE[1], CAM_LAYERS[1], x1, 0)))
        x2, im2, p2 = run(2); i2 = int(p2.argmax()); c2 = STAGES[2]["classes"][i2]
        rows.append(row(2, c2, p2[i2]))
        last_probs, last_classes = p2, STAGES[2]["classes"]
        if c2 == "COVID-19":
            final, conf = "COVID-19", p2[i2]
            cams.append((f"Stage 2: evidence for {final}", im2, grad_cam(PIPE[2], CAM_LAYERS[2], x2, i2)))
        else:
            x3, im3, p3 = run(3); i3 = int(p3.argmax()); c3 = STAGES[3]["classes"][i3]
            rows.append(row(3, c3, p3[i3]))
            last_probs, last_classes = p3, STAGES[3]["classes"]
            final, conf = c3, p3[i3]
            cams.append((f"Stage 3: evidence for {final}", im3, grad_cam(PIPE[3], CAM_LAYERS[3], x3, i3)))

    healthy = final.startswith("Normal")
    verdict = (f"## {'✅' if healthy else '⚠️'} Prediction: **{final}**  \n"
               f"Confidence of the deciding stage: **{conf*100:.1f}%**")
    label = {c: float(p) for c, p in zip(last_classes, last_probs)}
    table = pd.DataFrame(rows, columns=["Stage", "Model", "Prediction", "Confidence",
                                        "Paper acc", "Our acc", "Paper F1", "Our F1"])
    base_im = im1
    cam_img = _cam_figure(base_im, cams, f"Prediction: {final} ({conf*100:.1f}% confidence)",
                          "darkgreen" if healthy else "darkred") if cams else None
    return verdict, label, table, cam_img


# =====================================================================================
# PAPER vs OURS  (all metrics: table + charts)
# =====================================================================================
def metrics_table():
    rows = []
    for k in (1, 2, 3):
        for key in METRIC_KEYS:
            p, o = _m("paper", k, key), _m("ours", k, key)
            d = "-" if (p is None or o is None) else f"{o - p:+.2f}"
            rows.append([f"Stage {k} - {STAGES[k]['name']}", key, _fmt(p), _fmt(o), d])
    return pd.DataFrame(rows, columns=["Stage", "Metric", "Paper", "Ours (test)", "Ours - Paper"])


def split_table():
    return pd.DataFrame([[f"Stage {k} - {STAGES[k]['name']}"] + [f"{v:.2f}%" for v in
                          METRICS["train_val_test_acc"].get(S3 if k == 3 else str(k), [np.nan] * 3)]
                         for k in (1, 2, 3)], columns=["Stage", "Train acc", "Validation acc", "Test acc"])


def per_class_table():
    d = METRICS["ours_stage3_per_class"].get(S3, {})
    return pd.DataFrame([[c, f"{v[0]:.2f}%", f"{v[1]:.2f}%", f"{v[2]:.2f}%"] for c, v in d.items()],
                        columns=["Stage 3 class", "Precision", "Recall", "F1-score"])


def metrics_chart():
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharey=True)
    xs = np.arange(len(METRIC_KEYS)); w = 0.38
    for ax, k in zip(axes, (1, 2, 3)):
        p = [_m("paper", k, m) for m in METRIC_KEYS]; o = [_m("ours", k, m) for m in METRIC_KEYS]
        for i, (pv, ov) in enumerate(zip(p, o)):
            if pv is not None:
                ax.bar(i - w/2, pv, w, color="tab:gray", label="Paper" if i == 0 else None)
                ax.text(i - w/2, pv + 0.5, f"{pv:.1f}", ha="center", fontsize=7)
            if ov is not None:
                ax.bar(i + w/2, ov, w, color="tab:blue", label="Ours" if i == 0 else None)
                ax.text(i + w/2, ov + 0.5, f"{ov:.1f}", ha="center", fontsize=7)
        ax.set_xticks(xs); ax.set_xticklabels(METRIC_KEYS, rotation=30, fontsize=8)
        ax.set_title(f"Stage {k}: {STAGES[k]['name']}"); ax.grid(axis="y", alpha=.3); ax.set_ylim(60, 105)
    axes[0].set_ylabel("%"); h, l = axes[0].get_legend_handles_labels()
    by = dict(zip(l, h)); fig.legend(by.values(), by.keys(), loc="upper right")
    fig.suptitle("Paper vs Ours (bars missing = paper value not entered)"); fig.tight_layout()
    return _fig_to_array(fig)


# =====================================================================================
# GUI
# =====================================================================================
DISCLAIMER = ("*Research prototype - not a medical diagnosis. The heat-map shows where the model looked "
              "(red = strong influence); it is not a verified segmentation of the disease.*")

with gr.Blocks(title="Chest X-ray 3-Stage Pipeline", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🫁 3-Stage Chest X-ray Classifier\n"
                "`Normal?` → **SqueezeNet** → if abnormal → **ResNet-50** (COVID-19?) → if not → "
                "**EfficientNet-b0** (Bacterial pneumonia / Viral pneumonia / Tuberculosis)")
    with gr.Tab("🔍 Predict"):
        with gr.Row():
            with gr.Column(scale=1):
                inp = gr.Image(type="pil", label="Upload chest X-ray (JPG / PNG)")
                btn = gr.Button("Predict", variant="primary")
            with gr.Column(scale=1):
                verdict = gr.Markdown()
                label = gr.Label(label="Class probabilities (deciding stage)")
        table = gr.Dataframe(label="Stage-by-stage result  |  Paper vs Our accuracy", interactive=False)
        cam_out = gr.Image(label="Grad-CAM - region that drove the decision")
        gr.Markdown(DISCLAIMER)
        btn.click(predict, inp, [verdict, label, table, cam_out])
        inp.change(predict, inp, [verdict, label, table, cam_out])
    with gr.Tab("📊 Paper vs Ours - all metrics"):
        gr.Markdown("### Accuracy, Precision, Recall, F1, Specificity, AUC - base paper vs our models")
        gr.Dataframe(metrics_table(), interactive=False)
        gr.Image(metrics_chart(), show_label=False)
        gr.Markdown("### Our Train / Validation / Test accuracy")
        gr.Dataframe(split_table(), interactive=False)
        gr.Markdown("### Our Stage-3 per-class test results")
        gr.Dataframe(per_class_table(), interactive=False)
        gr.Markdown(
            "**Paper:** *Detection of COVID-19 & non-COVID diseases from chest X-rays using deep learning-based CNNs*, "
            "Soft Computing 30 (2026). Accuracy = 99 / 98 / 97 % (abstract); Recall = 1 - FNR from Sec. 7. "
            "Precision / F1 / Specificity / AUC are in the paper's Table 4 - add them to `metrics.json` "
            "(next to the weights) and restart the app.  \n"
            "**Not strictly comparable:** the paper tested on 100 images per stage; ours uses a 70/15/15 split "
            "(thousands of test images) and Stage 3 is a patient-level split. Stage-3 metrics are macro-averages.")


if __name__ == "__main__":
    if "google.colab" in sys.modules:
        demo.launch(share=True, debug=False)          # public link (valid while the Colab session lives)
    else:
        demo.launch(inbrowser=True)                   # your own PC -> opens http://127.0.0.1:7860 in the browser


Mounted at /content/drive
Stage 1: SqueezeNet       loaded from /content/drive/MyDrive/CXR_pipeline/results/stage1_SqueezeNet_best.pt
Stage 2: ResNet-50        loaded from /content/drive/MyDrive/CXR_pipeline/results/stage2_ResNet-50_best.pt
Stage 3: EfficientNet-b0  loaded from /content/drive/MyDrive/CXR_pipeline/results/stage3v2_efficientnet_b0_best.pt
metrics loaded from /content/drive/MyDrive/CXR_pipeline/results/metrics.json
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://52a46da0dc12ce2d0d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
